In [1]:
import os
import json
import math
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Iterable, Union, List, Dict
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm


# =========================================================
# Config
# =========================================================
CONFIG = {
    "processed_dir": r"C:\Users\tlsdn\Downloads\processed",   # train/val/test 폴더가 여기 아래 있어야 함
    "model_name": "emilyalsentzer/Bio_ClinicalBERT",
    "max_length": 256,
    "max_notes": 5,                         # admission당 최대 note 수
    "batch_size": 4,
    "num_workers": 0,
    "epochs": 5,
    "lr_encoder": 2e-5,
    "lr_head": 1e-3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout": 0.2,
    "lstm_hidden": 256,
    "lstm_layers": 1,
    "bidirectional": False,
    "top_k_icd": 100,
    "top_k_cpt": 100,
    "seed": 42,
    "threshold_search": True,
    "save_dir": "./outputs_icd_cpt",
    "use_discharge_narrative": False,       # 필요시 True
    "include_event_type_prefix": True,      # [radiology] ... 형태로 note에 prefix
    "gradient_clip": 1.0,
    "amp": True,
}


# =========================================================
# Reproducibility
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


# =========================================================
# Utilities
# =========================================================
def split_paths(processed_dir: Union[str, Path], split: str) -> List[Path]:
    split_dir = Path(processed_dir) / split
    if not split_dir.exists():
        raise FileNotFoundError(f"{split_dir} does not exist")
    paths = sorted(split_dir.glob("part-*.jsonl"))
    if not paths:
        raise FileNotFoundError(f"No part-*.jsonl found under {split_dir}")
    return paths


def iter_records(paths: Iterable[Path]):
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)


def build_label_vocab(paths: List[Path], label_field: str, top_k: Union[int, None] = None) -> List[str]:
    counter = Counter()
    for rec in iter_records(paths):
        codes = rec.get("labels", {}).get(label_field, [])
        counter.update(codes)
    items = counter.most_common(top_k) if top_k else counter.most_common()
    return [code for code, _ in items]


def multihot(codes: List[str], vocab_to_idx: Dict[str, int]) -> np.ndarray:
    y = np.zeros(len(vocab_to_idx), dtype=np.float32)
    for c in codes:
        idx = vocab_to_idx.get(c)
        if idx is not None:
            y[idx] = 1.0
    return y


def precision_at_k(y_score: np.ndarray, y_true: np.ndarray, k: int = 5) -> float:
    order = np.argsort(y_score, axis=1)[:, ::-1][:, :k]
    vals = []
    for i in range(len(order)):
        topk = order[i]
        vals.append(np.mean(y_true[i, topk]))
    return float(np.mean(vals)) if vals else 0.0


def best_micro_f1_threshold(y_score: np.ndarray, y_true: np.ndarray) -> float:
    eps = 1e-8
    flat_score = y_score.ravel()
    flat_true = y_true.ravel()
    precision, recall, thresholds = precision_recall_curve(flat_true, flat_score)
    f1 = 2 * precision * recall / (precision + recall + eps)

    if len(thresholds) == 0:
        return 0.5

    best_idx = np.argmax(f1[:-1]) if len(f1) > len(thresholds) else np.argmax(f1)
    best_idx = min(best_idx, len(thresholds) - 1)
    return float(thresholds[best_idx])


def compute_metrics(y_score: np.ndarray, y_true: np.ndarray, threshold: float, prefix: str) -> dict:
    y_pred = (y_score >= threshold).astype(np.float32)

    result = {
        f"{prefix}_f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}_f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}_p@5": precision_at_k(y_score, y_true, k=5),
        f"{prefix}_p@8": precision_at_k(y_score, y_true, k=8),
        f"{prefix}_threshold": threshold,
    }

    has_pos = y_true.sum(axis=0) > 0
    if has_pos.any():
        try:
            result[f"{prefix}_auc_micro"] = roc_auc_score(y_true[:, has_pos], y_score[:, has_pos], average="micro")
            result[f"{prefix}_auc_macro"] = roc_auc_score(y_true[:, has_pos], y_score[:, has_pos], average="macro")
        except Exception:
            result[f"{prefix}_auc_micro"] = None
            result[f"{prefix}_auc_macro"] = None
    else:
        result[f"{prefix}_auc_micro"] = None
        result[f"{prefix}_auc_macro"] = None

    return result


# =========================================================
# Dataset
# =========================================================
NOTE_EVENT_TYPES = {
    "ed_note",
    "admission_note",
    "nursing",
    "radiology",
    "ecg",
    "pharmacy",
    "respiratory",
}


class AdmissionDataset(Dataset):
    def __init__(
        self,
        paths: list[Path],
        icd_vocab: list[str],
        cpt_vocab: list[str],
        max_notes: int = 5,
        use_discharge_narrative: bool = False,
        include_event_type_prefix: bool = True,
    ):
        self.records = list(iter_records(paths))
        self.icd_to_idx = {c: i for i, c in enumerate(icd_vocab)}
        self.cpt_to_idx = {c: i for i, c in enumerate(cpt_vocab)}
        self.max_notes = max_notes
        self.use_discharge_narrative = use_discharge_narrative
        self.include_event_type_prefix = include_event_type_prefix

    def __len__(self):
        return len(self.records)

    def _extract_notes(self, rec: Dict[str, Any]) -> List[str]:
        notes = []
        events = rec.get("events", [])

        for ev in events:
            et = ev.get("event_type")
            if et not in NOTE_EVENT_TYPES:
                continue
            txt = ev.get("text")
            if not txt:
                continue

            if self.include_event_type_prefix:
                txt = f"[{et}] {txt}"

            notes.append(txt)

        notes = notes[: self.max_notes]

        if self.use_discharge_narrative:
            dn = rec.get("discharge_narrative", "")
            if dn:
                notes.append("[discharge_narrative] " + dn)

        if len(notes) == 0:
            notes = ["[empty_note]"]

        return notes[: self.max_notes]

    def __getitem__(self, idx):
        rec = self.records[idx]
        labels = rec.get("labels", {})

        notes = self._extract_notes(rec)
        icd_y = multihot(labels.get("icd10", []), self.icd_to_idx)
        cpt_y = multihot(labels.get("cpt", []), self.cpt_to_idx)

        return {
            "hadm_id": int(rec["hadm_id"]),
            "notes": notes,
            "icd_labels": torch.tensor(icd_y, dtype=torch.float32),
            "cpt_labels": torch.tensor(cpt_y, dtype=torch.float32),
        }


class Collator:
    def __init__(self, tokenizer, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        batch_size = len(batch)
        note_counts = [len(x["notes"]) for x in batch]
        max_notes_in_batch = max(note_counts)

        flat_notes = []
        note_mask = torch.zeros(batch_size, max_notes_in_batch, dtype=torch.long)

        for i, item in enumerate(batch):
            notes = item["notes"]
            for j, note in enumerate(notes):
                flat_notes.append(note)
                note_mask[i, j] = 1

        enc = self.tokenizer(
            flat_notes,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "hadm_ids": [x["hadm_id"] for x in batch],
            "input_ids": enc["input_ids"],              # (sum_notes, L)
            "attention_mask": enc["attention_mask"],    # (sum_notes, L)
            "note_counts": torch.tensor(note_counts, dtype=torch.long),
            "note_mask": note_mask,                     # (B, Nmax)
            "icd_labels": torch.stack([x["icd_labels"] for x in batch]),
            "cpt_labels": torch.stack([x["cpt_labels"] for x in batch]),
        }


# =========================================================
# Model
# =========================================================
class TemporalICDCPTModel(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_icd_labels: int,
        num_cpt_labels: int,
        lstm_hidden: int = 256,
        lstm_layers: int = 1,
        dropout: float = 0.2,
        bidirectional: bool = False,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
        )

        final_dim = lstm_hidden * self.num_directions

        self.icd_head = nn.Linear(final_dim, num_icd_labels)
        self.cpt_head = nn.Linear(final_dim, num_cpt_labels)

    def encode_notes(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]   # (sum_notes, H)
        return cls_vec

    def forward(self, input_ids, attention_mask, note_counts):
        """
        input_ids:      (sum_notes, L)
        attention_mask: (sum_notes, L)
        note_counts:    (B,)
        """
        note_vecs = self.encode_notes(input_ids, attention_mask)   # (sum_notes, H)

        batch_size = note_counts.size(0)
        max_notes = int(note_counts.max().item())
        H = note_vecs.size(-1)

        padded = note_vecs.new_zeros((batch_size, max_notes, H))
        lengths = []

        start = 0
        for i, n in enumerate(note_counts.tolist()):
            padded[i, :n] = note_vecs[start : start + n]
            lengths.append(n)
            start += n

        packed = nn.utils.rnn.pack_padded_sequence(
            padded, lengths=lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, (h_n, c_n) = self.lstm(packed)

        if self.bidirectional:
            # 마지막 layer의 forward/backward concat
            last_hidden = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        else:
            last_hidden = h_n[-1]

        rep = self.dropout(last_hidden)
        icd_logits = self.icd_head(rep)
        cpt_logits = self.cpt_head(rep)

        return icd_logits, cpt_logits


# =========================================================
# Train / Eval
# =========================================================
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="train", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        note_counts = batch["note_counts"].to(DEVICE)
        icd_labels = batch["icd_labels"].to(DEVICE)
        cpt_labels = batch["cpt_labels"].to(DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=CONFIG["amp"] and DEVICE.type == "cuda"):
            icd_logits, cpt_logits = model(input_ids, attention_mask, note_counts)

            loss_icd = F.binary_cross_entropy_with_logits(icd_logits, icd_labels)
            loss_cpt = F.binary_cross_entropy_with_logits(cpt_logits, cpt_labels)
            loss = loss_icd + loss_cpt

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, icd_threshold=0.5, cpt_threshold=0.5, search_threshold=False):
    model.eval()

    icd_scores_all = []
    cpt_scores_all = []
    icd_true_all = []
    cpt_true_all = []

    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        note_counts = batch["note_counts"].to(DEVICE)

        icd_logits, cpt_logits = model(input_ids, attention_mask, note_counts)
        icd_scores = torch.sigmoid(icd_logits).cpu().numpy()
        cpt_scores = torch.sigmoid(cpt_logits).cpu().numpy()

        icd_scores_all.append(icd_scores)
        cpt_scores_all.append(cpt_scores)
        icd_true_all.append(batch["icd_labels"].cpu().numpy())
        cpt_true_all.append(batch["cpt_labels"].cpu().numpy())

    icd_scores_all = np.concatenate(icd_scores_all, axis=0)
    cpt_scores_all = np.concatenate(cpt_scores_all, axis=0)
    icd_true_all = np.concatenate(icd_true_all, axis=0)
    cpt_true_all = np.concatenate(cpt_true_all, axis=0)

    if search_threshold:
        icd_threshold = best_micro_f1_threshold(icd_scores_all, icd_true_all)
        cpt_threshold = best_micro_f1_threshold(cpt_scores_all, cpt_true_all)

    metrics = {}
    metrics.update(compute_metrics(icd_scores_all, icd_true_all, icd_threshold, prefix="icd"))
    metrics.update(compute_metrics(cpt_scores_all, cpt_true_all, cpt_threshold, prefix="cpt"))

    return metrics, icd_scores_all, cpt_scores_all, icd_true_all, cpt_true_all


# =========================================================
# Decode predictions
# =========================================================
def decode_predictions(scores: np.ndarray, vocab: List[str], threshold: float) -> List[List[str]]:
    out = []
    for row in scores:
        codes = [vocab[i] for i, s in enumerate(row) if s >= threshold]
        out.append(codes)
    return out


# =========================================================
# Main
# =========================================================
def main():
    os.makedirs(CONFIG["save_dir"], exist_ok=True)

    train_paths = split_paths(CONFIG["processed_dir"], "train")
    val_paths = split_paths(CONFIG["processed_dir"], "val")
    test_paths = split_paths(CONFIG["processed_dir"], "test")

    print("Building vocab from TRAIN only...")
    icd_vocab = build_label_vocab(train_paths, "icd10", top_k=CONFIG["top_k_icd"])
    cpt_vocab = build_label_vocab(train_paths, "cpt", top_k=CONFIG["top_k_cpt"])

    print(f"ICD vocab size: {len(icd_vocab)}")
    print(f"CPT vocab size: {len(cpt_vocab)}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

    train_ds = AdmissionDataset(
        train_paths,
        icd_vocab=icd_vocab,
        cpt_vocab=cpt_vocab,
        max_notes=CONFIG["max_notes"],
        use_discharge_narrative=CONFIG["use_discharge_narrative"],
        include_event_type_prefix=CONFIG["include_event_type_prefix"],
    )
    val_ds = AdmissionDataset(
        val_paths,
        icd_vocab=icd_vocab,
        cpt_vocab=cpt_vocab,
        max_notes=CONFIG["max_notes"],
        use_discharge_narrative=CONFIG["use_discharge_narrative"],
        include_event_type_prefix=CONFIG["include_event_type_prefix"],
    )
    test_ds = AdmissionDataset(
        test_paths,
        icd_vocab=icd_vocab,
        cpt_vocab=cpt_vocab,
        max_notes=CONFIG["max_notes"],
        use_discharge_narrative=CONFIG["use_discharge_narrative"],
        include_event_type_prefix=CONFIG["include_event_type_prefix"],
    )

    collator = Collator(tokenizer, max_length=CONFIG["max_length"])

    train_loader = DataLoader(
        train_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        collate_fn=collator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        collate_fn=collator,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        collate_fn=collator,
    )

    model = TemporalICDCPTModel(
        model_name=CONFIG["model_name"],
        num_icd_labels=len(icd_vocab),
        num_cpt_labels=len(cpt_vocab),
        lstm_hidden=CONFIG["lstm_hidden"],
        lstm_layers=CONFIG["lstm_layers"],
        dropout=CONFIG["dropout"],
        bidirectional=CONFIG["bidirectional"],
    ).to(DEVICE)

    encoder_params = list(model.encoder.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": CONFIG["lr_encoder"]},
            {"params": head_params, "lr": CONFIG["lr_head"]},
        ],
        weight_decay=CONFIG["weight_decay"],
    )

    total_steps = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["amp"] and DEVICE.type == "cuda")

    best_val_icd = -1.0
    best_path = os.path.join(CONFIG["save_dir"], "best_model.pt")

    for epoch in range(1, CONFIG["epochs"] + 1):
        print(f"\n===== Epoch {epoch} / {CONFIG['epochs']} =====")
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        print(f"train_loss: {train_loss:.4f}")

        val_metrics, _, _, _, _ = evaluate(
            model,
            val_loader,
            search_threshold=CONFIG["threshold_search"],
        )

        print("VAL:", {k: round(v, 4) if isinstance(v, float) else v for k, v in val_metrics.items()})

        if val_metrics["icd_f1_micro"] > best_val_icd:
            best_val_icd = val_metrics["icd_f1_micro"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": CONFIG,
                    "icd_vocab": icd_vocab,
                    "cpt_vocab": cpt_vocab,
                    "val_metrics": val_metrics,
                },
                best_path,
            )
            print(f"Saved best model to {best_path}")

    print("\nLoading best model...")
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    icd_vocab = ckpt["icd_vocab"]
    cpt_vocab = ckpt["cpt_vocab"]

    # val threshold 다시 탐색
    val_metrics, _, _, _, _ = evaluate(model, val_loader, search_threshold=True)
    icd_thr = val_metrics["icd_threshold"]
    cpt_thr = val_metrics["cpt_threshold"]

    test_metrics, icd_scores, cpt_scores, icd_true, cpt_true = evaluate(
        model,
        test_loader,
        icd_threshold=icd_thr,
        cpt_threshold=cpt_thr,
        search_threshold=False,
    )

    print("\nTEST:", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_metrics.items()})

    # 예측 코드 decode 저장
    icd_pred_codes = decode_predictions(icd_scores, icd_vocab, icd_thr)
    cpt_pred_codes = decode_predictions(cpt_scores, cpt_vocab, cpt_thr)

    out_path = os.path.join(CONFIG["save_dir"], "test_predictions.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for i, item in enumerate(test_ds.records):
            row = {
                "hadm_id": item["hadm_id"],
                "pred_icd10": icd_pred_codes[i],
                "pred_cpt": cpt_pred_codes[i],
            }
            f.write(json.dumps(row) + "\n")

    print(f"Saved predictions to {out_path}")


if __name__ == "__main__":
    main()

main()

DEVICE: cuda
Building vocab from TRAIN only...
ICD vocab size: 100
CPT vocab size: 100


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_25340\2552594217.py:534: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` 


===== Epoch 1 / 5 =====


train:   0%|          | 0/9405 [00:00<?, ?it/s]C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_25340\2552594217.py:367: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CONFIG["amp"] and DEVICE.type == "cuda"):


...

VAL: {'icd_f1_micro': 0.4207, 'icd_f1_macro': 0.2924, 'icd_p@5': 0.4408, 'icd_p@8': 0.3668, 'icd_threshold': 0.2296, 'icd_auc_micro': 0.8061, 'icd_auc_macro': 0.769, 'cpt_f1_micro': 0.5706, 'cpt_f1_macro': 0.1846, 'cpt_p@5': 0.4821, 'cpt_p@8': 0.3798, 'cpt_threshold': 0.2933, 'cpt_auc_micro': 0.9593, 'cpt_auc_macro': 0.8725}
Saved best model to ./outputs_icd_cpt\best_model.pt

===== Epoch 5 / 5 =====


train:   0%|          | 0/9405 [00:00<?, ?it/s]C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_25340\2552594217.py:367: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CONFIG["amp"] and DEVICE.type == "cuda"):


train_loss: 0.2686


VAL: {'icd_f1_micro': 0.424, 'icd_f1_macro': 0.3111, 'icd_p@5': 0.4418, 'icd_p@8': 0.3692, 'icd_threshold': 0.2108, 'icd_auc_micro': 0.8025, 'icd_auc_macro': 0.7707, 'cpt_f1_micro': 0.5731, 'cpt_f1_macro': 0.1943, 'cpt_p@5': 0.4825, 'cpt_p@8': 0.3816, 'cpt_threshold': 0.3089, 'cpt_auc_micro': 0.9596, 'cpt_auc_macro': 0.8732}
Saved best model to ./outputs_icd_cpt\best_model.pt

Loading best model...



TEST: {'icd_f1_micro': 0.4345, 'icd_f1_macro': 0.3202, 'icd_p@5': 0.4501, 'icd_p@8': 0.3788, 'icd_threshold': 0.2108, 'icd_auc_micro': 0.8183, 'icd_auc_macro': 0.7779, 'cpt_f1_micro': 0.571, 'cpt_f1_macro': 0.1955, 'cpt_p@5': 0.4841, 'cpt_p@8': 0.3842, 'cpt_threshold': 0.3089, 'cpt_auc_micro': 0.9598, 'cpt_auc_macro': 0.8831}
Saved predictions to ./outputs_icd_cpt\test_predictions.jsonl


In [2]:
import json

path = "outputs_icd_cpt/test_predictions.jsonl"

with open(path, "r") as f:
    for i, line in enumerate(f):
        data = json.loads(line)

        print("HADM_ID:", data["hadm_id"])
        print("ICD:", data["pred_icd10"])
        print("CPT:", data["pred_cpt"])
        print("-"*50)

        if i == 4:
            break

HADM_ID: 100053
ICD: ['N179', 'J9600', 'J9690', 'A419', 'R6520', 'D696', 'I120', 'R6521']
CPT: ['99291', '99232', '99233', '94003', '99231', '94002', '99254', '99255', '99292', '36556', '90935']
--------------------------------------------------
HADM_ID: 100071
ICD: ['I10', 'I169', 'K219']
CPT: ['99291', '99232', '99233', '99239']
--------------------------------------------------
HADM_ID: 100074
ICD: ['N179', 'J9600', 'J9690', 'D62', 'E872', 'A419', 'R6520', 'R6521']
CPT: ['99291', '99232', '99233', '94003', '99231', '94002']
--------------------------------------------------
HADM_ID: 100104
ICD: ['J9600', 'J9690', 'A419', 'R6520']
CPT: ['99291', '99232', '99233', '94003', '99231', '94002', '36556']
--------------------------------------------------
HADM_ID: 100120
ICD: ['I10', 'I169', 'I509', 'I50814', 'I4891', 'N179', 'E119', 'J9600', 'J9690', 'N390', 'J449', 'J189', 'Z7901']
CPT: ['99291', '99232', '99233', '94003', '94002']
--------------------------------------------------


In [ ]:
import json
from pathlib import Path

processed_dir = r"C:\Users\tlsdn\Downloads\processed"
pred_path = "outputs_icd_cpt/test_predictions.jsonl"

with open(pred_path, "r") as f:
    preds = [json.loads(line) for line in f]

test_records = []

test_paths = sorted((Path(processed_dir) / "test").glob("part-*.jsonl"))

for p in test_paths:
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            test_records.append(json.loads(line))

# hadm_id -> record mapping
record_map = {rec["hadm_id"]: rec for rec in test_records}

for i in range(5):
    pred = preds[i]
    hadm_id = pred["hadm_id"]

    rec = record_map.get(hadm_id)

    print("HADM_ID:", hadm_id)

    print("\n[NOTES]")
    if rec:
        for ev in rec.get("events", []):
            txt = ev.get("text")
            if txt:
                print("-", txt[:200])   # 앞부분만 출력
    else:
        print("No record found")

    print("\nICD:", pred["pred_icd10"])
    print("CPT:", pred["pred_cpt"])

    print("="*60)

HADM_ID: 100053

[NOTES]
- [DEIDENTIFIED] 12:48 AM
 CHEST (PORTABLE AP)                                             Clip # [DEIDENTIFIED]
 Reason: pneumonia?  chf?
 ______________________________________________________________
- Focus Condition Update
See Flowsheet for Specific info

56 y/o male admitted from ED for tx of hypotension.  Pt came from [DEIDENTIFIED] [DEIDENTIFIED] rehab, he was transfered to [DEIDENTIFIED] [DEID
- [DEIDENTIFIED] 7:52 AM
 ABDOMEN U.S. (PORTABLE)                                         Clip # [DEIDENTIFIED]
 Reason: r/o portal venous thrombosis, assess ascitesPlease mark for
 Admitting Diagnosis:
- [DEIDENTIFIED] 4:43 PM
 CENTRAL LINE PLCT                                               Clip # [DEIDENTIFIED]
 Reason: please place line for HD, non-tunnelled line given elevated
 Admitting Diagnosis:
- CONDITION UPDATE
    PATIENT ORIENTED X 3 WHEN ASKED BUT OCCASIONALLY APPEARS CONFUSED- SPEAKING OF UNRELATED ISSUES AND TALKING WHEN NO ONE IN ROOM.  DENIES ANY

In [ ]:
import json
from pathlib import Path

processed_dir = r"C:\Users\tlsdn\Downloads\processed"
pred_path = "outputs_icd_cpt/test_predictions.jsonl"
output_txt = "outputs_icd_cpt/sample_predictions.txt"

with open(pred_path, "r") as f:
    preds = [json.loads(line) for line in f]

test_records = []
test_paths = sorted((Path(processed_dir) / "test").glob("part-*.jsonl"))

for p in test_paths:
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            test_records.append(json.loads(line))

record_map = {rec["hadm_id"]: rec for rec in test_records}

with open(output_txt, "w", encoding="utf-8") as out:

    for i in range(5):
        pred = preds[i]
        hadm_id = pred["hadm_id"]
        rec = record_map.get(hadm_id)

        out.write(f"HADM_ID: {hadm_id}\n")
        out.write("\n[NOTES]\n")

        if rec:
            for ev in rec.get("events", []):
                txt = ev.get("text")
                if txt:
                    out.write("- " + txt[:200] + "\n")
        else:
            out.write("No record found\n")

        out.write("\nICD: " + str(pred["pred_icd10"]) + "\n")
        out.write("CPT: " + str(pred["pred_cpt"]) + "\n")
        out.write("="*60 + "\n")

print("Saved to:", output_txt)

Saved to: outputs_icd_cpt/sample_predictions.txt


In [ ]:
import json
import csv

input_path = "outputs_icd_cpt/test_predictions.jsonl"
output_path = "outputs_icd_cpt/test_predictions.csv"

with open(input_path, "r") as f, open(output_path, "w", newline="", encoding="utf-8") as out:

    writer = csv.writer(out)
    
    # header
    writer.writerow(["hadm_id", "icd_codes", "cpt_codes"])

    for line in f:
        data = json.loads(line)

        hadm_id = data["hadm_id"]
        icd = ", ".join(data["pred_icd10"])   
        cpt = ", ".join(data["pred_cpt"])

        writer.writerow([hadm_id, icd, cpt])

print("CSV 저장 완료:", output_path)

CSV 저장 완료: outputs_icd_cpt/test_predictions.csv


In [17]:
import json
import csv

input_path = "outputs_icd_cpt/test_predictions.jsonl"
output_path = "outputs_icd_cpt/test_predictions.csv"

with open(input_path, "r") as f, open(output_path, "w", newline="", encoding="utf-8") as out:

    writer = csv.writer(out)
    
    # header
    writer.writerow(["hadm_id", "icd_codes", "cpt_codes"])

    for line in f:
        data = json.loads(line)

        hadm_id = data["hadm_id"]
        icd = ", ".join(data["pred_icd10"])   
        cpt = ", ".join(data["pred_cpt"])

        writer.writerow([hadm_id, icd, cpt])

print("CSV 저장 완료:", output_path)